In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

df_bronze = spark.table("taxi_catalog.bronze.yellow_taxi_raw")
display(df_bronze.limit(5))

In [0]:
df_bronze.select(
    F.count("*").alias("total_rows"),
    F.sum(F.col("tpep_pickup_datetime").isNull().cast("int")).alias("null_pickup"),
    F.sum(F.col("tpep_dropoff_datetime").isNull().cast("int")).alias("null_dropoff"),
    F.sum(F.col("trip_distance").isNull().cast("int")).alias("null_distance"),
    F.sum(F.col("total_amount").isNull().cast("int")).alias("null_total")
).show()

In [0]:
df_silver = (
    df_bronze
    .filter(F.col("tpep_pickup_datetime").isNotNull())
    .filter(F.col("tpep_dropoff_datetime").isNotNull())
    .filter(F.col("tpep_dropoff_datetime") >= F.col("tpep_pickup_datetime"))
    .filter(F.col("trip_distance").isNotNull() & (F.col("trip_distance") >= 0))
    .filter(F.col("fare_amount").isNotNull())
    .filter(F.col("total_amount").isNotNull())
)

In [0]:
df_silver = (
    df_silver
    .withColumn(
        "trip_duration_minutes",
        (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60
    )
    .withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
    .withColumn("pickup_year", F.year("tpep_pickup_datetime"))
    .withColumn("pickup_month", F.month("tpep_pickup_datetime"))
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .filter((F.col("trip_duration_minutes") > 0) & (F.col("trip_duration_minutes") <= 24 * 60))
)

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS taxi_catalog.silver")

(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("taxi_catalog.silver.yellow_taxi_clean")
)
print("Silver table written successfully!")

In [0]:
bronze_count = spark.table("taxi_catalog.bronze.yellow_taxi_raw").count()
silver_count = spark.table("taxi_catalog.silver.yellow_taxi_clean").count()

print("Bronze rows:", bronze_count)
print("Silver rows:", silver_count)
print("Filtered out:", bronze_count - silver_count)